[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB-B_python_ml_refresher.ipynb)

# NB-B · Bridge notebook: Python and machine-learning refresher
**Artificial Intelligence in Medical Imaging** · optional self-study, about 45 minutes, any time before Day 2

| | |
|---|---|
| For whom | Students who are rusty on NumPy, pandas or the train/validation/test idea. Engineers can skim; clinicians and Athens students: do it with a buddy |
| Runtime | CPU is enough. **No downloads, no deep learning**: it uses the 8×8 digit images that ship with scikit-learn |
| Checkpoints | Each section ends with assertions; if they pass, move on |

**What you will be able to do afterwards.** Read an image as an array (shape, dtype, the `uint8` trap) · make a stratified train/val/test split · tell a baseline from a model · recognise overfitting · see how leakage inflates a score · explain why 90% accuracy can be useless.

Nothing here is medical on purpose: digits are the simplest images that behave like images. The ideas carry over to chest X-rays unchanged.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
SEED = 42; rng = np.random.default_rng(SEED)
digits = load_digits()
print("images:", digits.images.shape, "| dtype:", digits.images.dtype, "| value range:", digits.images.min(), "to", digits.images.max())

## 1 · An image is an array
A grey image is a 2-D array: rows × columns, one number per pixel. A batch is 3-D: images × rows × columns. A colour image adds a last axis of 3. **Always print the shape first.**

In [ ]:
img = digits.images[0]
print("one image:", img.shape, "| batch:", digits.images.shape)
fig, ax = plt.subplots(1, 6, figsize=(9, 1.8))
for a, i in zip(ax, range(6)): a.imshow(digits.images[i], cmap="gray_r"); a.set_title(f"label {digits.target[i]}"); a.axis("off")
plt.show()
print(img.astype(int))

**TODO 1.** Three small tasks on `img`:
(a) the **mean** of the pixels in the centre 4×4 block (rows 2–5, columns 2–5, remember Python slicing is `start:stop` with `stop` excluded);
(b) the image **flipped left-right** (`img[:, ::-1]`);
(c) the image **scaled to 0–1** (divide by the maximum possible value, 16).

Then see the `uint8` trap.

In [ ]:
centre_mean = None     # TODO 1a: mean of the centre 4x4 block
flipped = None         # TODO 1b: flipped left-right
scaled = None          # TODO 1c: scaled to 0..1

In [ ]:
assert abs(centre_mean - img[2:6, 2:6].mean()) < 1e-12 and img[2:6, 2:6].size == 16
assert flipped.shape == img.shape and (flipped[:, 0] == img[:, -1]).all()
assert scaled.min() >= 0 and scaled.max() <= 1
# the uint8 trap: images are often stored as uint8 (0..255); arithmetic wraps around silently
a = np.array([200, 100], dtype=np.uint8); b = np.array([100, 100], dtype=np.uint8)
print("uint8: 200 + 100 =", (a + b)[0], "(wrapped!) | as int: ", (a.astype(int) + b.astype(int))[0])
assert (a + b)[0] == 44
print("Checkpoint 1 passed")

> **Why it matters in imaging.** A DICOM slice stored as `uint16` with a rescale slope, a PNG as `uint8`: subtracting, adding noise or averaging in the stored type gives wrong numbers with no error message. Convert to `float32` before you compute.

## 2 · A table of what you have (pandas)
Before modelling: how many images per class? A class table is the first thing a reviewer asks for.

In [ ]:
y = digits.target
table = pd.Series(y).value_counts().sort_index().rename("images").to_frame()
table["share"] = (table["images"] / len(y)).round(3)
table

In [ ]:
assert table["images"].sum() == len(y) == 1797 and table.shape[0] == 10
print("Checkpoint 2 passed: 10 classes, roughly balanced (about 10% each)")

## 3 · Three sets, and what each one is for
- **train**: the model learns from it
- **validation**: *you* choose things with it (model, depth, threshold)
- **test**: touched **once**, at the end

**TODO 2.** Make a stratified 60 / 20 / 20 split with `train_test_split` (twice) and a fixed seed. *Stratified* keeps the class shares equal in every set. Hint: first split off 40% (`test_size=0.4`), then split that part in half.

In [ ]:
X = digits.images.reshape(len(y), -1) / 16.0          # flatten each 8x8 image to 64 numbers
# TODO 2: X_tr, y_tr (60%), X_va, y_va (20%), X_te, y_te (20%), stratified, random_state=SEED
X_tr = X_va = X_te = y_tr = y_va = y_te = None

In [ ]:
n = len(y)
assert abs(len(y_tr) / n - 0.6) < 0.01 and abs(len(y_va) / n - 0.2) < 0.01 and abs(len(y_te) / n - 0.2) < 0.01, "sizes should be 60/20/20"
for s in (y_tr, y_va, y_te): assert np.abs(np.bincount(s) / len(s) - np.bincount(y) / n).max() < 0.02, "class shares should match the full set"
print("sizes:", len(y_tr), len(y_va), len(y_te), "| Checkpoint 3 passed")

## 4 · A baseline, then a model
A **baseline** is the dumbest thing that could work. If your model cannot beat it, you have learned nothing. Here: always predict the most common training class. Then logistic regression (scaling is part of the pipeline, so it is fitted on the training data only).

In [ ]:
majority = np.bincount(y_tr).argmax()
acc_base = accuracy_score(y_va, np.full_like(y_va, majority))
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
acc_val = accuracy_score(y_va, clf.predict(X_va))
print(f"baseline (always {majority}): {acc_base:.3f} | logistic regression: {acc_val:.3f}")
cm = confusion_matrix(y_va, clf.predict(X_va))
plt.imshow(cm, cmap="Blues"); plt.xlabel("predicted"); plt.ylabel("true"); plt.colorbar(); plt.title("confusion matrix (validation)"); plt.show()
assert acc_base < 0.2 and acc_val > 0.9
print("Checkpoint 4 passed")

## 5 · Overfitting, seen
A decision tree can memorise the training set if it is deep enough. Watch **train** and **validation** accuracy as the depth grows.

**TODO 3.** Fill the loop: fit a tree of each depth on the training set, record train and validation accuracy. Then pick the depth with the best **validation** accuracy (never the test set).

In [ ]:
depths = list(range(1, 16)); acc_tr_d, acc_va_d = [], []
for d in depths:
    # TODO 3a: fit DecisionTreeClassifier(max_depth=d, random_state=SEED) on the TRAINING set
    tree = None
    # TODO 3b: append its accuracy on train and on validation to acc_tr_d and acc_va_d  (tree.score(X, y))
# TODO 3c: best_depth = the depth with the highest VALIDATION accuracy
best_depth = None
plt.plot(depths, acc_tr_d, label="train"); plt.plot(depths, acc_va_d, label="validation"); plt.xlabel("tree depth"); plt.ylabel("accuracy"); plt.legend(); plt.show()

In [ ]:
assert len(acc_tr_d) == len(acc_va_d) == len(depths)
assert acc_tr_d[-1] > 0.99 and acc_tr_d[-1] - acc_va_d[-1] > 0.05, "the deepest tree should memorise the training set"
assert best_depth is not None and 3 <= best_depth <= 15
final = DecisionTreeClassifier(max_depth=best_depth, random_state=SEED).fit(X_tr, y_tr)
print(f"test accuracy (touched once): {final.score(X_te, y_te):.3f}")
print("Checkpoint 5 passed")

## 6 · Leakage in four lines
Take **pure noise**: 100 "patients", 5,000 random features, random labels. No model can do better than 50%. Now do what many people do by mistake: **select the 20 "best" features using all the data**, and only then cross-validate. The score looks real. It is not: the selection step already saw the test labels.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score
Xn = rng.normal(size=(100, 5000)); yn = rng.integers(0, 2, 100)
# WRONG: select on all data, then cross-validate
Xsel = SelectKBest(f_classif, k=20).fit_transform(Xn, yn)
wrong = cross_val_score(LogisticRegression(max_iter=1000), Xsel, yn, cv=5).mean()
# RIGHT: the selection is part of the pipeline, so it is refitted inside every fold
right = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000)), Xn, yn, cv=5).mean()
print(f"leaky cross-validation: {wrong:.2f}   |   correct cross-validation: {right:.2f}   (truth: 0.50)")
assert wrong > 0.75 and right < 0.65

> **Rule.** Anything that *learns from data* (scaling, feature selection, augmentation choices, a threshold) must be fitted on the training set only. In imaging, the classic version: several images of the same patient on both sides of the split.

## 7 · Why 90% accuracy can be useless
Make the problem **rare**: "is this digit a 9?" About 10% of the images are 9s.

**TODO 4.** Compute, for a model that **always says "not 9"**: accuracy, recall (of the 9s found) and precision (of the alarms that were right; undefined when there are no alarms: report 0). Then train a logistic regression and compute the same, by hand from the counts TP, FP, FN, TN.

In [ ]:
y9_tr, y9_va = (y_tr == 9).astype(int), (y_va == 9).astype(int)
always_no = np.zeros_like(y9_va)
# TODO 4a: accuracy, recall and precision of the "always no" model (use accuracy_score, recall_score, precision_score(..., zero_division=0))
acc0 = rec0 = prec0 = None
clf9 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y9_tr); pred9 = clf9.predict(X_va)
# TODO 4b: tp, fp, fn, tn from pred9 and y9_va, then recall, precision and accuracy by hand
tp = fp = fn = tn = 0
rec9 = prec9 = acc9 = None

In [ ]:
print(f"always 'not 9': accuracy {acc0:.3f}, recall {rec0:.3f}, precision {prec0:.3f}")
print(f"logistic regression: accuracy {acc9:.3f}, recall {rec9:.3f}, precision {prec9:.3f}   (TP {tp}, FP {fp}, FN {fn}, TN {tn})")
assert acc0 > 0.85 and rec0 == 0, "the useless model has high accuracy and finds no case"
assert abs(rec9 - recall_score(y9_va, pred9)) < 1e-9 and abs(acc9 - accuracy_score(y9_va, pred9)) < 1e-9
assert tp + fp + fn + tn == len(y9_va)
print("Checkpoint 6 passed")

**Reflection questions.**
1. Section 7: which number would you show a clinician, and why not accuracy?
2. Section 6: what would be the imaging version of "select features on all the data"?
3. Section 5: you chose the depth on validation. Why is it wrong to choose it on the test set "just to be sure"?
4. Section 1: you load a CT slice stored as `uint16` and add Gaussian noise of standard deviation 5. What can go wrong?

*Next:* NB03a (a first PyTorch network) or go straight to the day you are on.